# CF-conventions `xarray` datasets from the `waterdata` module

The `dataretrieval.waterdata.xarray` module mirrors the `waterdata` getters
(`get_daily`, `get_continuous`, `get_peaks`, `get_samples`, …) but returns a
[CF-conventions](https://cfconventions.org/) [`xarray.Dataset`](https://docs.xarray.dev/)
instead of a `pandas.DataFrame`. Units, statistics, and station coordinates come
straight from the data, and parameter names from a small cached lookup; all are
written onto the dataset as CF attributes, so the result is self-describing and
ready to write to netCDF.

This notebook covers:

1. the default **ragged** layout and how to read it;
2. **why** ragged is the default (and the `dense=True` opt-out);
3. the one trade-off you need to know — **selecting by time**;
4. multiple parameters in one pull;
5. water-quality samples (`get_samples`);
6. writing CF netCDF.

## Setup

The xarray helpers are an optional extra:

```bash
pip install dataretrieval[xarray]
```

As with the rest of `waterdata`, signing up for a free
[API key](https://api.waterdata.usgs.gov/signup/) gives you higher rate limits.
Import the xarray wrappers under a short alias so they don't shadow the plain
`DataFrame`-returning getters:

In [ ]:
import numpy as np

from dataretrieval.waterdata import xarray as wdx

## 1. A single time series

Pull a year of daily mean discharge (parameter `00060`, statistic `00003`) at one
gage. The wrapper takes the same arguments as `waterdata.get_daily`.

In [ ]:
ds = wdx.get_daily(
    monitoring_location_id="USGS-05407000",
    parameter_code="00060",
    statistic_id="00003",
    time="2023-01-01/2024-01-01",
)
ds

Note the shape of the result — this is a CF **contiguous ragged array**
(`featureType = "timeSeries"`):

* every observation lives along a single `obs` dimension;
* each *series* — one `(monitoring location, parameter, statistic)` — is one
  instance along the `timeseries` dimension;
* `row_size` records how many observations each series contributes (the CF
  `sample_dimension` link), and `timeseries_id` carries `cf_role`;
* because there is a single, homogeneous parameter here, the descriptors land
  directly on `value` (`long_name`, `units`, `cell_methods`, `standard_name`).

The flag columns (`qualifier`, `approval_status`) are linked as
`ancillary_variables`, and dataset attributes carry provenance:

In [ ]:
print(ds["value"].attrs)
print({k: ds.attrs[k] for k in ("Conventions", "featureType", "references", "date_modified")})

## 2. Why ragged is the default

Real collections are *ragged*: some gages have a century of record, others a few
years. Pull discharge at two gages with very different start dates and look at
`row_size`:

In [ ]:
sites = ["USGS-05407000", "USGS-02238500"]  # records since 1913 vs 2008
# Bound the window so the docs build stays light; the start-date gap still
# shows up as different row_size (~24 vs ~16 years of daily values).
ragged = wdx.get_daily(
    monitoring_location_id=sites,
    parameter_code="00060",
    statistic_id="00003",
    time="2000-01-01/2024-01-01",
)
print("dims      :", dict(ragged.sizes))
print("row_size  :", dict(zip(ragged["monitoring_location_id"].values, ragged["row_size"].values)))

The alternative is a **dense** `(monitoring_location_id, time)` grid — one named
variable per parameter, NaN where a series has no observation. It is convenient
(see the next section) but pays for a union time axis and NaN fill. Pass
`dense=True` to get it, and compare the in-memory size:

In [ ]:
dense = wdx.get_daily(
    monitoring_location_id=sites,
    parameter_code="00060",
    statistic_id="00003",
    time="2000-01-01/2024-01-01",
    dense=True,
)
print("dense dims :", dict(dense.sizes))
print("dense var  :", list(dense.data_vars))
print(f"ragged {ragged.nbytes/1e6:6.2f} MB   dense {dense.nbytes/1e6:6.2f} MB")

With only two gages the gap is modest, but it grows fast: a whole state's daily
discharge can be 15–30% dense, where the gridded layout balloons to hundreds of
megabytes (mostly NaN) while the ragged array stores only real observations.
That is why ragged is the default.

## 3. The trade-off: selecting by time

This is the one thing to internalize about the ragged layout.

In the **dense** dataset, `time` is a real dimension with an index, so
label-based selection just works:

In [ ]:
# all sites on a given day, addressed as a (site, time) grid
dense["discharge"].sel(time="2020-06-01")

In the **ragged** dataset, `value` is one flat list along `obs`, and `time` is
*just another variable* riding along `obs` — not a dimension. So `value` cannot
be addressed as a `(site, time)` grid, and a time selection returns whatever
observations happen to match, **mixed across sites with no labels** (and
identical dates across sites collide onto the same axis):

In [ ]:
print("value dims:", ragged["value"].dims)  # ('obs',) -- not (site, time)
# .sel(time=...) on the flat obs axis can't give you a per-series slice
ragged["value"].sel(time="2020-06-01")

To recover the convenient, time-indexed view you first **regroup** the flat
`obs` back into per-series pieces using the offsets implied by `row_size`. A
tiny helper:

In [ ]:
def series(ds, i):
    """Instance ``i`` of a ragged dataset as a time-indexed DataArray."""
    starts = np.concatenate([[0], np.cumsum(ds["row_size"].values)])
    sl = slice(int(starts[i]), int(starts[i + 1]))
    da = ds["value"].isel(obs=sl)
    return da.assign_coords(time=ds["time"].isel(obs=sl)).swap_dims(obs="time")


s0 = series(ragged, 0)
print(ragged["timeseries_id"].values[0])
s0.sel(time="2020-06-01")  # now .sel(time=...) works on a single series

For anything beyond a one-off slice, the [`cf-xarray`](https://cf-xarray.readthedocs.io/)
package understands the CF ragged encoding (`cf_role`, `sample_dimension`) and
will regroup/decode for you. Or, when you know the pull is small and overlapping,
just ask for `dense=True` up front and use `.sel(time=...)` directly.

**Rule of thumb:** ragged for storage and large multi-site pulls; `dense=True`
for ergonomic time-based slicing of a few overlapping series.

## 4. Multiple parameters in one pull

When a pull mixes parameters (and units), the ragged layout keeps a single
`value` and records the parameter/unit per *instance* — so nothing is
mislabeled. The homogeneous descriptors drop off `value` and live on the
`parameter_code` / `unit_of_measure` coordinates instead:

In [ ]:
multi = wdx.get_daily(
    monitoring_location_id="USGS-05407000",
    parameter_code=["00060", "00045"],  # discharge + precipitation
    time="2023-06-01/2023-07-01",
)
print("value long_name:", multi["value"].attrs.get("long_name"))
print("per-instance parameter_code:", multi["parameter_code"].values)
print("per-instance unit_of_measure:", multi["unit_of_measure"].values)

In the `dense=True` form each parameter instead becomes its own named variable
(`discharge`, `precipitation`, …) on the shared time grid.

## 5. Water-quality samples

`get_samples` returns discrete water-quality results in the same ragged shape:
one instance per `(monitoring location, characteristic, sample fraction)`, with
the result value plus `detection_condition` and `status` as ancillary
(censoring) variables. Characteristics are free text, so no CF `standard_name`
is inferred and non-numeric results coerce to NaN (the `detection_condition`
variable preserves non-detects).

In [ ]:
wq = wdx.get_samples(
    service="results",
    profile="basicphyschem",
    monitoringLocationIdentifier="USGS-05406500",
    activityStartDateLower="2019-01-01",
    activityStartDateUpper="2020-01-01",
)
print("dims:", dict(wq.sizes))
print("characteristics:", sorted(set(wq["characteristic"].values))[:8], "...")
print("ancillary:", wq["value"].attrs.get("ancillary_variables"))

## 6. Writing CF netCDF

Because the dataset already carries CF metadata in the standard ragged-array
encoding, it serializes straight to a self-describing netCDF file that
CF-aware tools (THREDDS, `cf-xarray`, Panoply, …) can read back (requires a
netCDF backend, e.g. `pip install netCDF4`):

In [ ]:
import os
import tempfile

path = os.path.join(tempfile.mkdtemp(), "daily_discharge.nc")
ragged.to_netcdf(path)  # CF-1.11 contiguous ragged array
print("wrote", path)

## References

* CF conventions — [Discrete Sampling Geometries](https://cfconventions.org/cf-conventions/cf-conventions.html#discrete-sampling-geometries)
  (contiguous ragged array representation)
* [`cf-xarray`](https://cf-xarray.readthedocs.io/) — decode/group CF DSG datasets
* [`xarray`](https://docs.xarray.dev/) documentation